In [1]:
!pip install ultralytics

# Download and extract the new dataset
!mkdir -p datasets
!curl -L "https://app.roboflow.com/ds/UhTLPdWWfY?key=xrEPF9NKoz" > roboflow.zip
!unzip -o roboflow.zip -d datasets
!rm roboflow.zip

# Define a helper class to mimic the dataset object for the training cell
class Dataset:
    def __init__(self, location):
        self.location = location

# Set the location to the new datasets folder
dataset = Dataset(location="./datasets")
print("New dataset downloaded and extracted into ./datasets")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   928  100   928    0     0   2451      0 --:--:-- --:--:-- --:--:--  2448
100 20.5M  100 20.5M    0     0  17.5M      0  0:00:01  0:00:01 --:--:-- 17.5M
Archive:  roboflow.zip
  inflating: datasets/README.roboflow.txt  
  inflating: datasets/data.yaml      
 extracting: datasets/train/images/000021_jpg.rf.HNoUfiJ4vevLXeEq65q2.jpg  
 extracting: datasets/train/images/000026_jpg.rf.HCgZWA26DjTmdWyZ74pI.jpg  
 extracting: datasets/train/images/000030_jpg.rf.ZciQSkAFi7QnpJcmnkRq.jpg  
 extracting: datasets/train/images/000046_jpg.rf.wtwwZqX7oKBR46BiW6Ll.jpg  
 extracting: datasets/train/images/000050_jpg.rf.nF5hP6tRqdfU4zbIVaQN.jpg  
 extracting: datasets/train/images/000050_jpg.rf.u25VGhcg8Y5zVslQ6cJy.jpg  
 extracting: datasets/train/images/000067_jpg.rf.6YCtc4GWKhACjpS8Tw6C.jpg  
 extracting: datasets/train/images/000067_jpg.r

In [ ]:
import os
import torch
import yaml
from pathlib import Path
try:
    import torch_xla.core.xla_model as xm
    tpu_available = True
except ImportError:
    tpu_available = False
from ultralytics import YOLO

# 1. SETUP & VERIFICATION
try:
    data_yaml_path = "./datasets/data.yaml"
    current_dir = os.getcwd()
    dataset_root = os.path.join(current_dir, "datasets")

    with open(data_yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    # Verification Print
    print("--- Dataset Verification ---")
    print(f"Project Name: {config.get('roboflow', {}).get('project', 'Unknown')}")
    print(f"Classes: {config.get('names')}")
    print("----------------------------")

    config['path'] = dataset_root
    config['train'] = "train/images"
    config['val'] = "train/images"
    config['test'] = "train/images"

    with open(data_yaml_path, 'w') as f:
        yaml.dump(config, f)
    print(f"Dataset paths corrected in: {data_yaml_path}")
except Exception as e:
    print(f"Error preparing dataset: {e}")

# 2. HARDWARE CHECK
if tpu_available:
    device = xm.xla_device()
    print(f"Using TPU: {device}")
elif torch.cuda.is_available():
    device = 0
else:
    device = "cpu"

# 3. PRODUCTION TRAINING
model = YOLO("yolo11s-obb.pt")
if os.path.exists(data_yaml_path):
    final_results = model.train(
        data=data_yaml_path,
        epochs=150,
        imgsz=640,
        batch=64 if tpu_available else (64 if torch.cuda.is_available() else 16),
        optimizer='AdamW',
        device=device,
        project="Car_Damage_OBB",
        name="production_run",
        task='obb',
        exist_ok=True,
        amp=False
    )
    print("Training finished.")

--- Dataset Verification ---
Project Name: satyas-workspace-mrftd
Classes: ['dent', 'scratch']
----------------------------
Dataset paths corrected in: ./datasets/data.yaml


/tmp/ipykernel_16521/434230301.py:40: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


Using TPU: xla:0
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./datasets/data.yaml, degrees=0.0, deterministic=True, device=xla:0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=production_run, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False

In [3]:
!ls -R /content/datasets

/content/datasets:
data.yaml  README.roboflow.txt	train

/content/datasets/train:
images	labels

/content/datasets/train/images:
000021_jpg.rf.HNoUfiJ4vevLXeEq65q2.jpg
000026_jpg.rf.HCgZWA26DjTmdWyZ74pI.jpg
000030_jpg.rf.ZciQSkAFi7QnpJcmnkRq.jpg
000046_jpg.rf.wtwwZqX7oKBR46BiW6Ll.jpg
000050_jpg.rf.nF5hP6tRqdfU4zbIVaQN.jpg
000050_jpg.rf.u25VGhcg8Y5zVslQ6cJy.jpg
000067_jpg.rf.6YCtc4GWKhACjpS8Tw6C.jpg
000067_jpg.rf.b5jhoqwlQvr0GtDOcf4G.jpg
000078_jpg.rf.rNVTHgFd4n7wjlCiyQwo.jpg
000121_jpg.rf.pA1LVukx82IHv8jAK5OF.jpg
000121_jpg.rf.x1B4SDlR3TKPGx7XKrTu.jpg
000147_jpg.rf.HuQWpNQVx2DTvDARzLvz.jpg
000364_jpg.rf.LnGbA9QCaAVc6Q9E7xTN.jpg
000368_jpg.rf.DyVccTscitXUzCQZTNrq.jpg
000368_jpg.rf.Zg693mdpYyElv9UN1XgK.jpg
000369_jpg.rf.TiJSlpZw4e31ZS0MG67C.jpg
000377_jpg.rf.iYuUr7Q2fkTXeoo2DW6p.jpg
000377_jpg.rf.QpseKDgKDFSffvCssV9l.jpg
0003_JPEG_jpg.rf.4cHFzyNBDQ0xX8qkaDpP.jpg
000449_jpg.rf.HNrzXL7d4k1bMWIjA5pL.jpg
000450_jpg.rf.8dXS3CeDCqW3vEl2ZJu4.jpg
000455_jpg.rf.CdJuBn6nlyqz1E6yF7Dh.jpg
000455_jpg

In [9]:
!ls -R /content

/content:
data.yaml  README.roboflow.txt	runs  sample_data  train  yolo11s-obb.pt

/content/runs:
obb

/content/runs/obb:
Car_Damage_OBB

/content/runs/obb/Car_Damage_OBB:
production_run

/content/runs/obb/Car_Damage_OBB/production_run:
args.yaml  weights

/content/runs/obb/Car_Damage_OBB/production_run/weights:

/content/sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md

/content/train:
images	labels

/content/train/images:
100_jpg.rf.2WrMrgj7jwISJ9bRPwhx.jpg
100_jpg.rf.kIIwSDjlEBHVAaGSLL6p.jpg
101_jpg.rf.Bh3wDccbt1ULWYDVd3mw.jpg
101_jpg.rf.vOilitQkFHP1o6uByu62.jpg
102_jpg.rf.kguKJfj54nOll24CD3zt.jpg
102_jpg.rf.V0huc4rfaTJC9laQLzbN.jpg
102_jpg.rf.zn6Cxw0DOCALq9xIvxIj.jpg
103_jpg.rf.iYyH8o10F5p13sgmKWUD.jpg
103_jpg.rf.TJvMmoKGHA7HuZOcjMFr.jpg
104_jpg.rf.qDNG906iiKitNjEPph15.jpg
104_jpg.rf.SD70JgbsJ1upEOY8AT48.jpg
104_jpg.rf.weKgsYKnNNypKytbdi7F.jpg
105_jpg.rf.0pgiuSyiz1IAKHlWvDFL.jpg
105_jpg.rf.b